## Objectives

1. Build a compact transformer-style classifier.
2. Implement full fine-tuning with AdamW.
3. Compare frozen embedding vs trainable embedding regimes.
4. Evaluate with accuracy and confusion analysis.

## Recommended Notebook Pattern

Structure the notebook as:

1. Setup
2. Data generation or loading
3. Model definition
4. Training
5. Evaluation
6. Comparison and interpretation

## Lab Validation Standard

For each major stage, include at least one validation output such as:

- dataset size or class balance
- model output shape
- loss or accuracy trace
- confusion matrix
- short comparison note across fine-tuning settings

## Setup


In [ ]:
#| eval: false
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


## Synthetic Financial Headline Dataset


In [ ]:
#| eval: false
def generate_headline_data(n=1800, seq_len=16, vocab_size=120):
    X, y = [], []
    for _ in range(n):
        label = torch.randint(0, 3, (1,)).item()
        seq = torch.randint(4, vocab_size, (seq_len,))
        if label == 0:
            seq[0] = 1  # positive marker
        elif label == 1:
            seq[0] = 2  # negative marker
        else:
            seq[0] = 3  # neutral marker
        X.append(seq)
        y.append(label)
    return TensorDataset(torch.stack(X), torch.tensor(y))

dataset = generate_headline_data()
train_ds, val_ds = torch.utils.data.random_split(dataset, [1400, 400])
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=64)


## Model


In [ ]:
#| eval: false
class TinyTransformerClassifier(nn.Module):
    def __init__(self, vocab_size=120, embed_dim=48, num_heads=4, num_classes=3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 2 * embed_dim),
            nn.ReLU(),
            nn.Linear(2 * embed_dim, embed_dim),
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        h = self.embed(x)
        attn_out, _ = self.attn(h, h, h)
        h = self.norm1(h + attn_out)
        h = self.norm2(h + self.ffn(h))
        cls = h[:, 0, :]
        return self.classifier(cls)


## Training Utility


In [ ]:
#| eval: false
def train_eval(model, train_dl, val_dl, epochs=6, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()

    history = {"train_acc": [], "val_acc": []}

    for ep in range(1, epochs + 1):
        model.train()
        correct = total = 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            correct += (logits.argmax(-1) == yb).sum().item()
            total += len(yb)
        tr = correct / total

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).argmax(-1)
                correct += (pred == yb).sum().item()
                total += len(yb)
        va = correct / total

        history["train_acc"].append(tr)
        history["val_acc"].append(va)
        print(f"Epoch {ep}: train_acc={tr:.3f}, val_acc={va:.3f}")

    return history


## Experiment A: Full Fine-Tuning


In [ ]:
#| eval: false
model_full = TinyTransformerClassifier()
h_full = train_eval(model_full, train_dl, val_dl, epochs=6, lr=1e-3)


## Experiment B: Frozen Embedding


In [ ]:
#| eval: false
model_frozen = TinyTransformerClassifier()
model_frozen.embed.weight.requires_grad = False
h_frozen = train_eval(model_frozen, train_dl, val_dl, epochs=6, lr=1e-3)


## Compare Curves


In [ ]:
#| eval: false
plt.plot(h_full["val_acc"], label="Val: Full FT")
plt.plot(h_frozen["val_acc"], label="Val: Frozen Emb")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Fine-Tuning Comparison")
plt.legend()
plt.tight_layout()
plt.show()


## Lab Questions

1. Which regime converges faster: full fine-tuning or frozen embedding?
2. Try learning rates {1e-4, 5e-4, 1e-3, 3e-3}. Which is best?
3. Increase model width (`embed_dim`) to 96. How do accuracy and runtime change?
4. Add dropout before the classifier. Does it improve validation performance?
5. Create a confusion matrix for the final model and interpret major error patterns.
